### EOF on multiple layers

In [2]:
#!/usr/bin/env python3
# ============================================================
# Combined EOF Analysis for depth-mean CMIP6 ocean temperature
# for the ATLANTIC (using already-masked files)
#
# For each model, this script:
#   - loads all thetao_masked_*.nc members
#   - computes depth-mean thetao for:
#         0–300 m, 0–1000 m, 1000–5000 m
#   - computes combined EOFs across members
#   - saves EOF, PC(member,time,mode), variance_fraction
#
# Works for:
#   EC-Earth3 (75 levels)
#   IPSL-CM6A-LR (75 levels)
#   CESM2 (60 levels)
# ============================================================

import os
import glob
import re
import numpy as np
import xarray as xr
from eofs.standard import Eof
import warnings
warnings.filterwarnings("ignore")

# ============================
# CONFIG (EDIT HERE)
# ============================
MODEL   = "EC-Earth3"   # ← CHANGE: "EC-Earth3", "IPSL-CM6A-LR", "CESM2"
VAR     = "thetao"
NMODES  = 10            # we only need the first 10 modes

# Input dirs (already Atlantic-masked θ)
if MODEL == "EC-Earth3":
    IN_DIR = "/data/projects/nckf/frekle/CMIP6_data/EC-Earth3/thetao/"
elif MODEL == "IPSL-CM6A-LR":
    IN_DIR = "/data/projects/nckf/frekle/CMIP6_data/IPSL-CM6A-LR/thetao/"
elif MODEL == "CESM2":
    IN_DIR = "/data/projects/nckf/frekle/CMIP6_data/CESM2/thetao/"
else:
    raise ValueError("Unknown MODEL")

OUT_DIR = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/combined_depthmeans/"
os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 1850
END_YEAR   = 2014

# depth ranges (in meters)
DEPTH_RANGES = [
    (0.0, 300.0),
    (0.0, 1000.0),
    (1000.0, 5000.0),
]

# ============================
# 1. UNIVERSAL GRID STANDARDIZATION
# ============================

def standardize_latlon(ds):
    """
    Standardize lat/lon to 2D arrays with dims ('y','x') for ALL grid types:
    - Regular grid:   lat(lat), lon(lon)   → broadcast to 2D
    - Curvilinear:    lat(j,i), lon(j,i)   → rename dims to ('y','x')
    Removes EC-Earth3 bogus coordinate variables named 'x'/'y'.
    """

    # detect lat/lon var names
    lat_var = next((v for v in ["lat", "latitude", "nav_lat"] if v in ds), None)
    lon_var = next((v for v in ["lon", "longitude", "nav_lon"] if v in ds), None)
    if lat_var is None or lon_var is None:
        raise KeyError("latitude/longitude not found")

    lat = ds[lat_var]
    lon = ds[lon_var]

    # ---------------------------------------------
    # CASE A — Regular grid (1D lat, 1D lon)
    # ---------------------------------------------
    if lat.ndim == 1 and lon.ndim == 1:
        print("   → Regular grid (1D lat/lon) detected")

        # broadcast into 2D
        lon2d, lat2d = np.meshgrid(lon.values, lat.values)

        # remove old lat/lon
        ds2 = ds.drop_vars([lat_var, lon_var])

        # rename dims first (safe)
        ds2 = ds2.rename_dims({lat.dims[0]: "y", lon.dims[0]: "x"})

        # add new 2D coords
        ds2["lat"] = xr.DataArray(lat2d, dims=("y", "x"))
        ds2["lon"] = xr.DataArray(lon2d, dims=("y", "x"))

        return ds2, ("y", "x")

    # ---------------------------------------------
    # CASE B — Curvilinear grid (2D lat, 2D lon)
    # ---------------------------------------------
    elif lat.ndim == 2 and lon.ndim == 2:
        print("   → Curvilinear grid (2D lat/lon) detected")

        ydim, xdim = lat.dims   # e.g. ("j","i")

        # swap_dims allows renaming even if ds has variables named 'x'/'y'
        ds2 = ds.swap_dims({ydim: "y", xdim: "x"})

        # rename lat/lon variables
        ds2 = ds2.rename({lat_var: "lat", lon_var: "lon"})

        # drop EC-Earth3 bogus vars
        for bad in ["x", "y"]:
            if bad in ds2 and bad not in ["lat", "lon"]:
                print(f"   ⚠ Dropping variable '{bad}' (conflicts with dims)")
                try:
                    ds2 = ds2.drop_vars(bad)
                except Exception:
                    pass

        return ds2, ("y", "x")

    else:
        raise ValueError("Unknown lat/lon grid format")

# ============================
# 2. MEMBER LABEL EXTRACTION
# ============================

def parse_member_label(path):
    base = os.path.basename(path)
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", base)
    if m:
        return m.group(1)
    return os.path.splitext(base)[0]

# ============================
# 3. FIND VERTICAL DIMENSION
# ============================

def get_depth_dim_and_coord(da, ds):
    """
    Detect vertical dimension/coordinate for thetao (75 or 60 levels).
    Returns (depth_dim_name, depth_coord_da).
    """
    for cand in ["olevel", "lev", "depth", "z"]:
        if cand in da.dims:
            return cand, ds[cand]
    raise KeyError("No recognized vertical dimension (olevel/lev/depth/z)")

# ============================
# 4. PROCESS: ONE DEPTH RANGE
# ============================

def run_depth_range(zmin, zmax):
    print(f"\n==============================")
    print(f"Depth range: {zmin:.0f}–{zmax:.0f} m")
    print(f"==============================\n")

    # -----------------------------------------------------------------
    # 4.1. Load all members and compute depth-mean anomalies per member
    # -----------------------------------------------------------------
    pattern = os.path.join(IN_DIR, f"{VAR}_masked_r*i*p*f*.nc")
    files = sorted(glob.glob(pattern))
    if not files:
        # fallback: maybe just r* pattern
        pattern = os.path.join(IN_DIR, f"{VAR}_masked_r*.nc")
        files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f"No masked {VAR} files for {MODEL} in {IN_DIR}")

    print(f"📂 Found {len(files)} members for {MODEL}\n")

    anom_list = []
    member_labels = []
    lat2d = lon2d = None

    for idx, f in enumerate(files, start=1):
        label = parse_member_label(f)
        member_labels.append(label)

        print(f"→ Loading member {idx}/{len(files)}: {os.path.basename(f)} ({label})")
        ds = xr.open_dataset(f)

        # Standardize grid
        print("   Standardizing spatial dims…")
        ds, (yd, xd) = standardize_latlon(ds)
        if lat2d is None:
            lat2d = ds["lat"]
            lon2d = ds["lon"]

        da = ds[VAR]  # thetao

        # detect vertical dim
        depth_dim, depth_coord = get_depth_dim_and_coord(da, ds)

        # Subset time
        yrs = da["time"].dt.year
        start = START_YEAR if START_YEAR else int(yrs.min())
        end   = END_YEAR   if END_YEAR   else int(yrs.max())
        da = da.sel(time=slice(f"{start}-01-01", f"{end}-12-31"))

        # -----------------------------------------
        # Select depth levels in [zmin, zmax] m
        # -----------------------------------------
        depth_vals = depth_coord.values
        mask_depth = (depth_vals >= zmin) & (depth_vals <= zmax)

        if mask_depth.sum() == 0:
            raise ValueError(
                f"No depth levels found in {zmin}–{zmax} m for file {f}"
            )

        print(f"   Using {mask_depth.sum()} levels in this range")

        da_sel = da.isel({depth_dim: mask_depth})

        # depth-mean temperature (simple mean over chosen levels)
        da_meanz = da_sel.mean(dim=depth_dim)

        # compute anomaly (remove time mean)
        print("   Computing anomaly…")
        anom = da_meanz - da_meanz.mean("time")

        # add member dimension
        anom = anom.expand_dims(member=[label])
        anom_list.append(anom)

    # -----------------------------------------
    # 4.2. Align time across members
    # -----------------------------------------
    print("\n🧭 Finding common time axis…")

    common_time = anom_list[0]["time"].values
    for a in anom_list[1:]:
        common_time = np.intersect1d(common_time, a["time"].values)

    if common_time.size == 0:
        raise ValueError("No overlapping time range across members!")

    common_time = xr.DataArray(common_time, dims=("time",))
    print(f"   → Common timesteps: {common_time.size}\n")

    aligned = [a.sel(time=common_time) for a in anom_list]
    combined = xr.concat(aligned, dim="member")

    n_member = combined.sizes["member"]
    n_time   = combined.sizes["time"]
    ny, nx   = combined.sizes["y"], combined.sizes["x"]

    print(f"   → Combined shape: member={n_member}, time={n_time}, y={ny}, x={nx}")

    # -----------------------------------------
    # 4.3. Prepare data for EOF
    # -----------------------------------------
    print("\n🧱 Preparing for EOF solver…")
    # combined has dims (member, time, y, x)
    data = combined.transpose("member", "time", "y", "x").values
    data2d = data.reshape(n_member * n_time, ny * nx)
    data2d = np.ma.masked_invalid(data2d)

    # compute spatial weights (area or cos(lat))
    print("→ Computing weights…")
    ref_ds = xr.open_dataset(files[0])
    if "areacello" in ref_ds:
        area = ref_ds["areacello"]
        if area.ndim == 2:
            w = np.sqrt(area.values.reshape(ny * nx))
        else:
            w = np.sqrt(np.cos(np.deg2rad(lat2d.values)).reshape(ny * nx))
    else:
        w = np.sqrt(np.cos(np.deg2rad(lat2d.values)).reshape(ny * nx))

    # -----------------------------------------
    # 4.4. Run EOF
    # -----------------------------------------
    print("\n🚀 Running EOF solver…")
    solver = Eof(data2d, weights=w)

    EOFs = solver.eofs(neofs=NMODES).reshape(NMODES, ny, nx)
    PCs_flat = solver.pcs(npcs=NMODES)  # shape (n_member*n_time, NMODES)
    VF   = solver.varianceFraction()[0:NMODES]

    # reshape PCs back to (member, time, mode)
    PCs = PCs_flat.reshape(n_member, n_time, NMODES)

    # -----------------------------------------
    # 4.5. Save output
    # -----------------------------------------
    zlabel = f"{int(zmin)}-{int(zmax)}m"
    outfile = os.path.join(
        OUT_DIR,
        f"EOF_combined_{VAR}_depthmean_{zlabel}.nc"
    )

    print(f"\n💾 Saving output to:\n   {outfile}")

    ds_out = xr.Dataset(
        {
            "EOF": (("mode", "y", "x"), EOFs),
            "PC":  (("member", "time", "mode"), PCs),
            "variance_fraction": (("mode",), VF),
        },
        coords={
            "mode": np.arange(1, NMODES + 1),
            "member": np.array(member_labels, dtype=object),
            "time": common_time,
            "lat": lat2d,
            "lon": lon2d,
        },
    )

    ds_out.to_netcdf(outfile)
    print("\n🎉 DONE for this depth range.\n")


# ============================
# 5. RUN FOR ALL DEPTH RANGES
# ============================

if __name__ == "__main__":
    print(f"=== MODEL: {MODEL}, variable: {VAR} ===")
    for (zmin, zmax) in DEPTH_RANGES:
        run_depth_range(zmin, zmax)
    print("\n✅ All depth ranges processed for this model.\n")



=== MODEL: EC-Earth3, variable: thetao ===

Depth range: 0–300 m

📂 Found 22 members for EC-Earth3

→ Loading member 1/22: thetao_masked_r10i1p1f1.nc (r10i1p1f1)
   Standardizing spatial dims…
   → Curvilinear grid (2D lat/lon) detected
   ⚠ Dropping variable 'x' (conflicts with dims)
   ⚠ Dropping variable 'y' (conflicts with dims)
   Using 34 levels in this range
   Computing anomaly…
→ Loading member 2/22: thetao_masked_r11i1p1f1.nc (r11i1p1f1)
   Standardizing spatial dims…
   → Curvilinear grid (2D lat/lon) detected
   ⚠ Dropping variable 'x' (conflicts with dims)
   ⚠ Dropping variable 'y' (conflicts with dims)
   Using 34 levels in this range
   Computing anomaly…
→ Loading member 3/22: thetao_masked_r12i1p1f1.nc (r12i1p1f1)
   Standardizing spatial dims…
   → Curvilinear grid (2D lat/lon) detected
   ⚠ Dropping variable 'x' (conflicts with dims)
   ⚠ Dropping variable 'y' (conflicts with dims)
   Using 34 levels in this range
   Computing anomaly…
→ Loading member 4/22: thetao_